# Fine-tuning example

Compare TabPFN-2.5 before and after one epoch of fine-tuning on Churn, with model-specific
(`native`) and `shared` preprocessing. Run the cells in order after completing the
[setup](../README.md#setup) and selecting the project's `.venv` kernel.

In [ ]:
import os
import sys

import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import display

from tfm_finetuning.experiments.configuration import expand_runs
from tfm_finetuning.experiments.execution.notebook import load_comparison, run_command
from tfm_finetuning.utils.paths import get_paths, get_repo_root

root = get_repo_root()
load_dotenv(root / ".env", override=False)
results = (root / get_paths().results_root).resolve()

## 1. Settings

Change the dataset, seed or training budget below (`context_size` is the number of rows per
training chunk). Tracking defaults come from `.env`: `disabled` keeps local reports only,
`offline` writes local W&B logs and `online` uploads them.

In [ ]:
config = yaml.safe_load((root / "configs/examples/introductory.yaml").read_text())
config["datasets"] = ["churn"]
config["split"]["seeds"] = [0]
config["settings"]["finetune.epochs"] = 1
config["settings"]["finetune.context_size"] = 1024
config["parameters"]["preprocessing"]["values"] = ["native", "shared"]

mode = os.environ.get("WANDB_MODE", "disabled")  # online, offline or disabled
project = os.environ.get("WANDB_PROJECT", "tfm-finetuning")
entity = os.environ.get("WANDB_ENTITY", "")  # optional account/team
if mode not in {"online", "offline", "disabled"}:
    raise ValueError("Choose online, offline, or disabled.")
os.environ.update(WANDB_MODE=mode, WANDB_PROJECT=project)
config["wandb"] = {"project": project, "mode": mode}
if entity:
    os.environ["WANDB_ENTITY"] = entity
    config["wandb"]["entity"] = entity
else:
    os.environ.pop("WANDB_ENTITY", None)

## 2. Preview

Save the settings and list the planned runs (two by default). Nothing is downloaded yet.
To run the saved file from a terminal instead:
`uv run tfm-experiment --config results/notebook/introductory.yaml`.

In [ ]:
config_path = results / "notebook" / "introductory.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
command = [
    sys.executable, "-m", "tfm_finetuning.experiments.run_experiment",
    "--config", str(config_path), "--results-root", str(results),
]
runs = expand_runs(config, repo_root=root)
print(f"Saved configuration: {config_path}")
display(pd.DataFrame([
    {key: run[key] for key in ("dataset", "model", "preprocessing", "fold", "seed")}
    for run in runs
]))

## 3. Train

Downloads or verifies the TabPFN-2.5 weights (needs Hugging Face access via `HF_TOKEN` or a
saved login), then trains both variants. Missing datasets download automatically, and
completed runs are skipped on rerun.

In [ ]:
run_command([sys.executable, "-m", "tfm_finetuning.data.download.tabpfn_checkpoints", "--group", "v2.5"])
if mode == "online":
    import wandb

    if not wandb.login():
        raise RuntimeError("Authenticate with W&B or choose offline/disabled tracking.")
run_command(command)

## 4. Compare results

The table compares the original model (ICL), the validation-selected checkpoint and the last
checkpoint. Higher ROC-AUC is better, and selected epoch `0` means validation preferred the
original model. One epoch may not improve it. Choose settings on validation data and report
test data.

For the full thesis experiments, continue with [HYDRA experiments](hydra_experiments.ipynb).

In [ ]:
comparison = load_comparison(runs, results)
display(comparison.round(4))